# FaceVox — Landmark Transformer Training (Colab)

Train the FaceVox expression classifier with GPU acceleration.

1. Upload your `dataset_*.json` files to the file browser (left panel)
2. Run all cells
3. Download the `.pt` model file from `checkpoints/`

In [ ]:
!pip install torch numpy scikit-learn

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load Data

In [ ]:
def load_datasets(data_dir='.', prefixes=None):
    """Load and merge multiple dataset files."""
    if prefixes is None:
        prefixes = []
        for f in os.listdir(data_dir):
            if f.startswith('dataset_') and f.endswith('.json'):
                prefixes.append(f.replace('dataset_', '').replace('.json', ''))
        if not prefixes:
            prefixes = ['']
    
    all_data = []
    all_labels = []
    
    for prefix in prefixes:
        fname = f'dataset_{prefix}.json' if prefix else 'dataset.json'
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            print(f'  Skipping {path} (not found)')
            continue
        with open(path) as f:
            ds = json.load(f)
        n = len(ds['data'])
        all_data.extend(ds['data'])
        all_labels.extend(ds['labels'])
        print(f'  Loaded {fname}: {n} samples')
    
    has_raw = any('_raw_landmarks' in d for d in all_data)
    print(f'\nTotal: {len(all_data)} samples, has_raw_landmarks: {has_raw}')
    return all_data, all_labels, has_raw

data, labels, has_raw = load_datasets('.')

In [ ]:
FEATURE_NAMES = [
    'mouth_open', 'mouth_width', 'lip_height_avg',
    'left_ear', 'right_ear', 'ear_avg',
    'left_brow_height', 'right_brow_height',
    'pitch', 'yaw',
]

if has_raw:
    X = np.array([d['_raw_landmarks'] for d in data], dtype=np.float32)
    print(f'Using raw landmarks: {X.shape}')
else:
    X = np.array([[d.get(name, 0.0) for name in FEATURE_NAMES] for d in data], dtype=np.float32)
    print(f'Using extracted features: {X.shape}')

y = np.array(labels, dtype=np.int64)
num_classes = len(np.unique(y))
input_dim = X.shape[1]
print(f'Classes: {num_classes}, Input dim: {input_dim}')

## Model Definitions

In [ ]:
class OcclusionAttention(nn.Module):
    def __init__(self, d_model=128, nhead=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, occlusion_mask=None):
        attn_mask = ~occlusion_mask.bool() if occlusion_mask is not None else None
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=attn_mask)
        if occlusion_mask is not None:
            vis = occlusion_mask.unsqueeze(-1).float()
            hid = 1.0 - vis
            gate = self.gate(torch.cat([x, attn_out], dim=-1))
            x = x * hid + (gate * attn_out + (1 - gate) * x) * vis
        else:
            x = x + attn_out
        x = self.norm1(x + self.ffn(self.norm1(x)))
        return x


class LandmarkTransformer(nn.Module):
    def __init__(self, input_dim=1434, num_classes=7, d_model=128, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_embed = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.classifier = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model*2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model*2, num_classes))

    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1) + self.pos_embed
        return self.classifier(self.transformer(x).squeeze(1))


class OcclusionAwareClassifier(nn.Module):
    def __init__(self, input_dim=1434, num_landmarks=478, num_classes=7, d_model=128, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.num_landmarks = num_landmarks
        cpl = input_dim // num_landmarks
        self.landmark_embed = nn.Sequential(nn.Linear(cpl, d_model//4), nn.GELU(), nn.Linear(d_model//4, d_model))
        self.landmark_pos = nn.Parameter(torch.randn(1, num_landmarks, d_model) * 0.02)
        self.occlusion_attn = OcclusionAttention(d_model, nhead, dropout)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.readout_attn = nn.Linear(d_model, 1)
        self.classifier = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model*2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model*2, num_classes))

    def forward(self, x, visibility=None):
        B = x.shape[0]
        cpl = x.shape[1] // self.num_landmarks
        x = x.view(B, self.num_landmarks, cpl)
        x = self.landmark_embed(x) + self.landmark_pos
        x = self.occlusion_attn(x, visibility)
        x = self.transformer(x)
        w = F.softmax(self.readout_attn(x), dim=1)
        x = (x * w).sum(1)
        return self.classifier(x)

## Training

In [ ]:
MODEL_TYPE = 'occlusion_aware'  # 'spatial' or 'occlusion_aware'
EPOCHS = 80
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 0.01

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Val: {len(X_val)}')

if MODEL_TYPE == 'occlusion_aware' and has_raw:
    model = OcclusionAwareClassifier(input_dim=input_dim, num_classes=num_classes).to(device)
else:
    model = LandmarkTransformer(input_dim=input_dim, num_classes=num_classes).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val_acc = 0
best_state = None
train_losses = []
val_accs = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for bx, by in train_loader:
        logits = model(bx)
        loss = criterion(logits, by)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t).argmax(1)
        val_acc = (val_preds == y_val_t).float().mean().item()
    
    train_losses.append(total_loss / len(train_loader))
    val_accs.append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} — loss: {train_losses[-1]:.4f} val_acc: {val_acc:.3f}')

print(f'\nBest val accuracy: {best_val_acc:.3f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax2.plot(val_accs)
ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    y_pred = model(X_val_t).argmax(1).cpu().numpy()

id_to_name = {
    0: 'neutral', 1: 'happy', 2: 'sad', 3: 'surprised',
    4: 'angry', 5: 'disgusted', 6: 'fearful',
}
unique = sorted(np.unique(y_val))
target_names = [id_to_name.get(i, str(i)) for i in unique]

print(classification_report(y_val, y_pred, labels=unique, target_names=target_names))

cm = confusion_matrix(y_val, y_pred, labels=unique)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Export

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
torch.save({
    'model_state_dict': best_state,
    'input_dim': input_dim,
    'num_classes': num_classes,
    'model_type': MODEL_TYPE,
    'config': {
        'input_dim': input_dim,
        'num_classes': num_classes,
        'model_type': MODEL_TYPE,
    },
}, f'checkpoints/expression_{MODEL_TYPE}.pt')
print(f'Saved checkpoints/expression_{MODEL_TYPE}.pt')
print('Download this file and place it in your FaceVox checkpoints/ folder.')